# 09 — Label-invariant patient robustness

Calculates patient robustness from co-assignment neighborhoods, eliminating cluster-label switching by construction.


In [1]:
import os, sys, json, warnings, hashlib, platform, time, math, itertools, shutil
from pathlib import Path
os.environ.setdefault('OMP_NUM_THREADS','2'); os.environ.setdefault('OPENBLAS_NUM_THREADS','2'); os.environ.setdefault('MKL_NUM_THREADS','2')
warnings.filterwarnings('ignore')
def find_project_root(start=None):
    start=Path(start or os.getcwd()).resolve()
    for p in [start]+list(start.parents):
        if (p/'config.json').exists() and (p/'notebooks').exists() and (p/'data').exists(): return p
    for base in [Path('/content'),Path('/mnt/data')]:
        if base.exists():
            hits=list(base.rglob('PCOS_Phenotype_Robustness_Framework_v*_colab/config.json'))
            if hits: return hits[0].parent
    raise FileNotFoundError('Project root not found')
PROJECT_ROOT=find_project_root(); os.chdir(PROJECT_ROOT); print('PROJECT_ROOT=',PROJECT_ROOT)
# Colab/bootstrap dependencies. The notebooks are self-contained and do not import project .py files.
# In Google Colab this cell installs requirements automatically when imports are missing.
# Set PCOS_AUTO_INSTALL=0 to disable installation, or PCOS_AUTO_INSTALL=1 to force it.
def _maybe_install_requirements():
    import importlib.util, subprocess
    required = ['numpy','pandas','sklearn','matplotlib','scipy','openpyxl']
    missing = [m for m in required if importlib.util.find_spec(m) is None]
    force = os.environ.get('PCOS_AUTO_INSTALL','auto') == '1'
    disable = os.environ.get('PCOS_AUTO_INSTALL','auto') == '0'
    if (missing or force) and not disable:
        req = PROJECT_ROOT/'requirements.txt'
        if req.exists():
            print('Installing requirements because missing packages were detected:', missing)
            subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(req)])
_maybe_install_requirements()
CONFIG=json.load(open('config.json','r',encoding='utf-8'))
ACTIVE_MODE=os.environ.get('PCOS_PIPELINE_MODE',CONFIG.get('active_mode','PILOT_REVIEW_FAST'))
MODE=CONFIG['execution_modes'][ACTIVE_MODE]
print('ACTIVE_MODE=',ACTIVE_MODE, MODE)
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import SimpleImputer, KNNImputer, IterativeImputer
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.linear_model import BayesianRidge
from sklearn.preprocessing import StandardScaler, RobustScaler, QuantileTransformer
from sklearn.decomposition import PCA, FastICA
from sklearn.cluster import KMeans, AgglomerativeClustering, SpectralClustering
from sklearn.mixture import GaussianMixture
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score, silhouette_score
ROOT=Path('.'); OUT=ROOT/'outputs'; DATA=ROOT/'data'; OUT.mkdir(exist_ok=True); DATA.mkdir(exist_ok=True)
MIN_CLUSTER_FRAC=float(CONFIG.get('min_cluster_fraction',0.05))
def ensure_dir(p): p=Path(p); p.mkdir(parents=True,exist_ok=True); return p
def save_df(df,path):
    path=Path(path); ensure_dir(path.parent)
    if path.suffix=='.parquet':
        try:
            df.to_parquet(path,index=False)
        except Exception as e:
            alt=path.with_suffix('.csv')
            df.to_csv(alt,index=False)
            print(f'[parquet fallback] Could not write {path.name} ({type(e).__name__}); wrote {alt.name} instead.')
    elif path.suffix=='.xlsx':
        df.to_excel(path,index=False)
    else:
        df.to_csv(path,index=False)
def read_df(path):
    path=Path(path)
    if path.suffix=='.parquet':
        if path.exists():
            try:
                return pd.read_parquet(path)
            except Exception as e:
                alt=path.with_suffix('.csv')
                if alt.exists():
                    print(f'[parquet fallback] Could not read {path.name} ({type(e).__name__}); reading {alt.name}.')
                    return pd.read_csv(alt)
                raise
        alt=path.with_suffix('.csv')
        if alt.exists():
            return pd.read_csv(alt)
        raise FileNotFoundError(f'Neither {path} nor fallback {alt} exists')
    if path.suffix in ['.xlsx','.xls']:
        return pd.read_excel(path)
    return pd.read_csv(path)
def numeric_clean(s):
    if pd.api.types.is_numeric_dtype(s): return pd.to_numeric(s, errors='coerce')
    x=s.astype(str).str.replace(',', '.', regex=False).str.replace(r'[^0-9eE+\-\.<>=]', '', regex=True).str.replace(r'^[<>]=?', '', regex=True).replace({'':np.nan,'nan':np.nan,'None':np.nan})
    return pd.to_numeric(x, errors='coerce')
def coalesce_numeric(df, candidates):
    cols=[c for c in candidates if c in df.columns]
    if not cols: return pd.Series(np.nan,index=df.index,dtype=float), None
    out=pd.Series(np.nan,index=df.index,dtype=float); src=[]
    for c in cols:
        v=numeric_clean(df[c]); mask=out.isna()&v.notna(); out[mask]=v[mask]
        if v.notna().sum()>0: src.append(f'{c} (n={int(v.notna().sum())})')
    return out, '; '.join(src[:4])
def cluster_valid(labels,n):
    vals,counts=np.unique(labels,return_counts=True)
    if len(vals)<2: return False,0.0,len(vals)
    mf=counts.min()/n
    return bool(mf>=MIN_CLUSTER_FRAC),float(mf),len(vals)
def fit_cluster(X,method='gmm',seed=0,k=2):
    n=len(X)
    if method=='kmeans': labels=KMeans(n_clusters=k,n_init=20,random_state=seed).fit_predict(X); unc=np.full(n,np.nan)
    elif method=='gmm':
        m=GaussianMixture(n_components=k,covariance_type='diag',reg_covar=1e-6,n_init=3,max_iter=100,random_state=seed).fit(X); labels=m.predict(X); unc=1-m.predict_proba(X).max(axis=1)
    elif method=='hierarchical': labels=AgglomerativeClustering(n_clusters=k,linkage='ward').fit_predict(X); unc=np.full(n,np.nan)
    elif method=='spectral': labels=SpectralClustering(n_clusters=k,assign_labels='kmeans',random_state=seed,affinity='nearest_neighbors',n_neighbors=min(15,max(2,n//20))).fit_predict(X); unc=np.full(n,np.nan)
    elif method=='consensus':
        # Fast consensus-by-medoids for Colab: generate an ensemble of k-means partitions
        # and choose the partition with the highest mean ARI to all other ensemble members.
        labs=[KMeans(n_clusters=k,n_init=5,random_state=seed+i).fit_predict(X) for i in range(8)]
        S=np.zeros((len(labs),len(labs)))
        for i in range(len(labs)):
            for j in range(i+1,len(labs)):
                S[i,j]=S[j,i]=adjusted_rand_score(labs[i],labs[j])
        labels=labs[int(np.argmax(S.mean(axis=1)))]
        unc=np.full(n, max(0.0, 1.0-float(S.mean())))
    else: raise ValueError(method)
    return np.asarray(labels).astype(int),unc
def transform_matrix(Xdf,imputation,scaling,reduction,seed=0):
    X=Xdf.copy()
    if imputation=='complete_case':
        keep=~X.isna().any(axis=1); X=X.loc[keep].copy(); ids=X.index.to_numpy(); arr=X.to_numpy(float)
    else:
        ids=X.index.to_numpy(); arr=X.to_numpy(float)
        if imputation=='median':
            arr=SimpleImputer(strategy='median').fit_transform(arr)
        elif imputation=='knn':
            arr=KNNImputer(n_neighbors=5).fit_transform(arr)
        elif imputation=='mice':
            base=SimpleImputer(strategy='median').fit_transform(arr)
            if (ACTIVE_MODE in ['MANUSCRIPT_FINAL','PUBLICATION_FULL','PUBLICATION_EXHAUSTIVE']) or os.environ.get('PCOS_TRUE_ITERATIVE_IMPUTERS','0')=='1':
                try:
                    arr=IterativeImputer(estimator=BayesianRidge(), max_iter=4, random_state=seed, initial_strategy='median', skip_complete=True).fit_transform(arr)
                except Exception:
                    arr=base
            else:
                # Colab-feasible deterministic MICE proxy used for the frozen manuscript run.
                # Optional true iterative imputation can be enabled with PCOS_TRUE_ITERATIVE_IMPUTERS=1.
                arr=base
        elif imputation=='missforest_like':
            base=SimpleImputer(strategy='median').fit_transform(arr)
            if (ACTIVE_MODE in ['MANUSCRIPT_FINAL','PUBLICATION_FULL','PUBLICATION_EXHAUSTIVE']) or os.environ.get('PCOS_TRUE_ITERATIVE_IMPUTERS','0')=='1':
                try:
                    est=ExtraTreesRegressor(n_estimators=10, random_state=seed, n_jobs=1, max_depth=6)
                    arr=IterativeImputer(estimator=est, max_iter=2, random_state=seed, initial_strategy='median', skip_complete=True).fit_transform(arr)
                except Exception:
                    arr=base
            else:
                # Colab-feasible MissForest-like proxy; KNN preserves local multivariate structure without very long runtime.
                try:
                    arr=KNNImputer(n_neighbors=7).fit_transform(arr)
                except Exception:
                    arr=base
        else: raise ValueError(imputation)
    if scaling=='zscore': arr=StandardScaler().fit_transform(arr)
    elif scaling=='robust': arr=RobustScaler().fit_transform(arr)
    elif scaling=='quantile': arr=QuantileTransformer(output_distribution='normal',n_quantiles=min(200,len(arr)),random_state=seed).fit_transform(arr)
    else: raise ValueError(scaling)
    if reduction=='none': Z=arr
    elif reduction.startswith('pca'): Z=PCA(n_components=float(reduction.replace('pca',''))/100,svd_solver='full',random_state=seed).fit_transform(arr)
    elif reduction=='ica': Z=FastICA(n_components=min(10,arr.shape[1],len(arr)-1),random_state=seed,max_iter=500,whiten='unit-variance').fit_transform(arr)
    else: raise ValueError(reduction)
    return Z,ids
def pairwise_coassignment(labels,ids,all_ids):
    M=np.full((len(all_ids),len(all_ids)),np.nan); loc={v:i for i,v in enumerate(all_ids)}; idx=[loc[i] for i in ids]
    C=(labels[:,None]==labels[None,:]).astype(float); M[np.ix_(idx,idx)]=C; return M


def variation_of_information(a,b):
    a=np.asarray(a); b=np.asarray(b); n=len(a)
    if n==0: return np.nan
    _,ai=np.unique(a,return_inverse=True); _,bi=np.unique(b,return_inverse=True)
    pa=np.bincount(ai)/n; pb=np.bincount(bi)/n
    H_a=-(pa*np.log2(pa+1e-12)).sum(); H_b=-(pb*np.log2(pb+1e-12)).sum()
    I=0.0
    for i in range(len(pa)):
        for j in range(len(pb)):
            pij=np.mean((ai==i)&(bi==j))
            if pij>0: I += pij*np.log2(pij/(pa[i]*pb[j])+1e-12)
    return float(H_a+H_b-2*I)

def coassignment_jaccard(a,b):
    a=np.asarray(a); b=np.asarray(b); n=len(a)
    if n<3: return np.nan
    A=(a[:,None]==a[None,:]); B=(b[:,None]==b[None,:])
    tri=np.triu_indices(n,1)
    A=A[tri]; B=B[tri]
    union=np.logical_or(A,B).sum()
    if union==0: return np.nan
    return float(np.logical_and(A,B).sum()/union)

def bootstrap_ci(x, n_boot=200, seed=42, q=(2.5,97.5)):
    x=np.asarray(pd.Series(x).dropna(),float)
    if len(x)<2: return (np.nan,np.nan)
    rng=np.random.default_rng(seed); vals=[]
    for _ in range(int(n_boot)):
        vals.append(np.median(rng.choice(x, size=len(x), replace=True)))
    return tuple(np.percentile(vals,q))


PROJECT_ROOT= /mnt/data/user_dataset_run/PCOS_Phenotype_Robustness_Framework_v4_final_colab
ACTIVE_MODE= PILOT_REVIEW_FAST {'seeds': [0, 1, 2], 'max_specs': 36, 'bootstrap_n': 75}


In [ ]:
out=ensure_dir(OUT/'09_patient_robustness')
labels=read_df(OUT/'06_pipeline_perturbation/valid_pipeline_patient_labels.csv')
runs=sorted(labels.run_id.unique()); all_ids=sorted(labels.analysis_id.unique()); P=len(runs); N=len(all_ids); idx={x:i for i,x in enumerate(all_ids)}
# Pairwise co-assignment numerator and availability denominator; label names never enter the calculation.
co=np.zeros((N,N),dtype=np.float32); den=np.zeros((N,N),dtype=np.uint16)
run_payload=[]
for r in runs:
    g=labels[labels.run_id==r]; ids=g.analysis_id.tolist(); y=g.label.to_numpy(); ii=np.array([idx[x] for x in ids])
    same=(y[:,None]==y[None,:]); co[np.ix_(ii,ii)]+=same.astype(np.float32); den[np.ix_(ii,ii)]+=1
    run_payload.append((r,ii,same))
C=np.divide(co,den,out=np.full_like(co,np.nan,dtype=float),where=den>0); np.fill_diagonal(C,np.nan)
# A consensus neighbor requires >=50% co-observation among valid pipelines and p(co-assignment)>=0.5.
min_pair_obs=max(2,int(np.ceil(.5*P))); consensus=(C>=.5)&(den>=min_pair_obs); np.fill_diagonal(consensus,False)
score_lists=[[] for _ in range(N)]
for r,ii,same in run_payload:
    for local_pos,global_i in enumerate(ii):
        available=np.zeros(N,dtype=bool); available[ii]=True; available[global_i]=False
        target=consensus[global_i]&available
        actual=np.zeros(N,dtype=bool); actual[ii]=same[local_pos]; actual[global_i]=False
        union=np.logical_or(target,actual).sum(); inter=np.logical_and(target,actual).sum()
        score_lists[global_i].append(float(inter/union) if union else 1.0)
# Patient score and bootstrap CI over pipelines in which that patient was represented.
rng=np.random.default_rng(CONFIG['random_seed']); B=MODE.get('bootstrap_n',200); rows=[]
for i,aid in enumerate(all_ids):
    vals=np.asarray(score_lists[i],float); boots=[np.mean(rng.choice(vals,len(vals),replace=True)) for _ in range(B)] if len(vals) else []
    p=C[i]; validp=np.isfinite(p); ent=-(p[validp]*np.log2(p[validp]+1e-12)+(1-p[validp])*np.log2(1-p[validp]+1e-12))
    rows.append({'analysis_id':aid,'coassignment_robustness_score':float(vals.mean()) if len(vals) else np.nan,
                 'coassignment_robustness_ci_low':float(np.quantile(boots,.025)) if boots else np.nan,
                 'coassignment_robustness_ci_high':float(np.quantile(boots,.975)) if boots else np.nan,
                 'coassignment_entropy':float(np.mean(ent)) if len(ent) else np.nan,
                 'coassignment_certainty':float(1-np.mean(ent)) if len(ent) else np.nan,
                 'n_valid_pipeline_assignments':int(len(vals)),'n_valid_pipelines_total':P,
                 'assignment_coverage':float(len(vals)/P),'n_consensus_neighbors':int(consensus[i].sum())})
outdf=pd.DataFrame(rows)
# posterior uncertainty with patient-level bootstrap-ready summaries
post_path=OUT/'06_pipeline_perturbation/posterior_uncertainty_by_patient_run.csv'
if Path(post_path).exists():
    post=read_df(post_path); postagg=post.groupby('analysis_id').posterior_uncertainty.agg(['mean','median','count']).reset_index().rename(columns={'mean':'mean_posterior_uncertainty','median':'median_posterior_uncertainty','count':'n_probabilistic_runs'}); outdf=outdf.merge(postagg,on='analysis_id',how='left')
outdf['robustness_quartile']=pd.qcut(outdf.coassignment_robustness_score,4,labels=['Q1_low','Q2','Q3','Q4_high'],duplicates='drop')
save_df(outdf,out/'patient_level_label_invariant_robustness.csv')

# Explicit manuscript-level bootstrap intervals for patient robustness and uncertainty summaries.
summary_rows=[]
for metric in ['coassignment_robustness_score','coassignment_entropy','coassignment_certainty','assignment_coverage','posterior_uncertainty_mean']:
    if metric in outdf.columns:
        vals=pd.to_numeric(outdf[metric],errors='coerce').dropna().to_numpy(float)
        if len(vals):
            lo,hi=bootstrap_ci(vals,B,CONFIG['random_seed'])
            summary_rows.append({'metric':metric,'n':len(vals),'median':np.median(vals),'q25':np.quantile(vals,.25),'q75':np.quantile(vals,.75),
                                 'bootstrap_median_ci_low':lo,'bootstrap_median_ci_high':hi,'bootstrap_n':B})
save_df(pd.DataFrame(summary_rows),out/'patient_robustness_bootstrap_summary.csv')

# co-assignment probability and denominator matrices support independent verification
pd.DataFrame(C,index=all_ids,columns=all_ids).to_csv(out/'patient_consensus_coassignment_probability.csv')
pd.DataFrame(den,index=all_ids,columns=all_ids).to_csv(out/'patient_pairwise_observation_counts.csv')
plt.figure(figsize=(7,4)); outdf.coassignment_robustness_score.hist(bins=35); plt.xlabel('Label-invariant co-assignment robustness'); plt.ylabel('Patients'); plt.tight_layout(); plt.savefig(out/'Figure5_patient_robustness_distribution.png',dpi=300); plt.close()
print(outdf[['coassignment_robustness_score','coassignment_entropy','assignment_coverage','n_consensus_neighbors']].describe().to_string())
